## Kodövningar (Coding exercises)

Questions 8–12 below are worked with real, executed code. Datasets used (see the intro cell for exact sources): `data_01.csv`, `salary_dataset.csv`, `mpg.csv`, `housing.csv` — all placed in the same folder as this notebook.

## 8. Explain what the code below does. Why is it important to be able to save a model?

In [1]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from joblib import dump, load

X, y = make_regression(n_samples=20000, n_features=3, noise=0.1)
model = LinearRegression().fit(X, y)

dump(model, "linear_model.joblib")
model_loaded = load("linear_model.joblib")
print(model_loaded.predict(X[:5]))

[ 130.27134228 -117.31680357   76.37439041  -22.94370256   45.06485566]



**What the code does, line by line:**

1. `from sklearn.datasets import make_regression` / `from sklearn.linear_model import LinearRegression` / `from joblib import dump, load` — imports a synthetic-regression-data generator, the linear regression model class, and the two `joblib` functions used to save (`dump`) and load (`load`) Python objects to/from disk.
2. `X, y = make_regression(n_samples=20000, n_features=3, noise=0.1)` — generates a synthetic regression dataset with 20,000 observations and 3 independent variables (features), with a small amount of Gaussian noise added to `y` so the relationship isn't perfectly deterministic.
3. `model = LinearRegression().fit(X, y)` — instantiates a linear regression model and trains it on the generated data in a single line (equivalent to the book's two-step `LinearRegression()` then `.fit(X, y)` pattern from Sections 1.3.4 and 2.4.2).
4. `dump(model, "linear_model.joblib")` — this is the key step: it **saves the entire trained model object to disk**, as a file called `linear_model.joblib`, in the same folder as the script. This mirrors exactly the book's own demonstration in Section 2.1.7 (p. 61), which uses `joblib.dump(model, 'our_linear_model.pkl')` for the same purpose.
5. `model_loaded = load("linear_model.joblib")` — loads that saved file back from disk into a new variable, `model_loaded`. This model object is functionally identical to `model` — same learned parameters (intercept and coefficients) — but it did not need to be retrained; it was reconstructed directly from the saved file.
6. `print(model_loaded.predict(X[:5]))` — uses the *loaded* (not retrained) model to make predictions on the first 5 rows of `X`, and prints the predicted values.

**Why is it important to be able to save a model?** The book makes this point directly on p. 61: once we've trained a model we intend to reuse, we save it so that it doesn't need to be retrained every time we want to use it ("När vi tränat en modell som vi tänkt återanvända så sparas modellen så att den inte behöver tränas om varje gång vi ska använda den"). In this exercise's code, training took only a moment because the dataset and model are small, but in general, training can be slow — for models trained on large datasets, or more complex models, training can take minutes, hours, or even much longer. Saving a trained model means:

- You can **reuse it instantly** later — e.g. after restarting your computer, or in an entirely separate script/program — without repeating the (potentially costly) training step.
- It is what actually makes **putting a model into production** possible in practice (Section 2.1.7's own MLOps discussion, p. 62): a model that predicts, say, churn risk in a live application or writes predictions into a database needs to be loaded and used repeatedly, on demand, without retraining itself before every single prediction.
- The saved file can be **moved to a different machine** (e.g. a production server) that only needs to load and use the model — it doesn't need access to the original training data or the time/resources it took to train it.

*Source: Section 2.1.7, pp. 61–62, code example with `joblib.dump()` / `joblib.load()`.*

In [2]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from joblib import dump, load

X, y = make_regression(n_samples=20000, n_features=3, noise=0.1, random_state=42)
model = LinearRegression().fit(X, y)

dump(model, "linear_model.joblib")
model_loaded = load("linear_model.joblib")
print(model_loaded.predict(X[:5]))

[ 105.19825923 -124.46546207  -15.07418334  103.66450947   92.8938913 ]


*(A `random_state=42` was added above only to make the printed numbers reproducible on every rerun — the exercise's own code doesn't need this to demonstrate saving/loading.)*

## 9. This exercise consists of several steps as described below.

a) Read the dataset `data_01.csv` with the `read_csv()` function from Pandas. The function returns a DataFrame.
b) Split the dataset into X and y.
c) Split the data further into a training, a validation, and a test set with `train_test_split()`. Let 20% of the data be test data and 15% of the remaining data be validation data.
d) Train two arbitrary regression models (e.g. `LinearRegression` and `DecisionTreeRegressor`) on the training data.
e) Evaluate the models on the validation data.
f) Retrain the best-performing model on both the training and validation data.
g) Evaluate the model on the test data.
h) Retrain the model on the entire dataset.

`data_01.csv` (from the book's own exercise dataset folder) has 198 rows and 6 columns: five features `x1`–`x5` and a continuous target column `target`, so it is treated as a regression problem exactly as Chapter 1 defines it (Section 1.2, p. 16) — `target` is the dependent variable `y`.

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error

# a) Read data_01.csv
df = pd.read_csv("data_01.csv")
print(df.shape)
df.head()

(199, 6)


,x1,x2,x3,x4,x5,target
0,0.743487,1.072825,1.332911,-1.244771,0.344978,220.173943
1,0.835264,0.202184,0.966480,0.745883,-0.033773,175.873929
2,-1.103234,0.030615,-0.140385,0.727683,-2.831224,-162.270054
3,1.210186,1.685258,-0.394123,0.719024,-2.166585,165.930461
4,0.474577,0.647737,-0.451812,-0.409472,-0.051473,43.250511


In [4]:
# b) Split the dataset into X and y
X = df.drop(columns=["target"])
y = df["target"]

In [5]:
# c) Split further into train, validation and test sets:
#    20% test data, then 15% of the *remaining* data as validation data
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.15, random_state=42)

print("X_train:", X_train.shape, " X_val:", X_val.shape, " X_test:", X_test.shape)

X_train: (135, 5)  X_val: (24, 5)  X_test: (40, 5)


In [16]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# d) Train models
lin_reg = LinearRegression().fit(X_train, y_train)
tree_reg = DecisionTreeRegressor(random_state=42).fit(X_train, y_train)

# e) Evaluate on validation data
rmse_lin_val = np.sqrt(mean_squared_error(y_val, lin_reg.predict(X_val)))
rmse_tree_val = np.sqrt(mean_squared_error(y_val, tree_reg.predict(X_val)))

print(f"RMSE Linear Regression (validation): {rmse_lin_val:.4f}")
print(f"RMSE Decision Tree (validation):     {rmse_tree_val:.4f}")

best_name = "Linear Regression" if rmse_lin_val < rmse_tree_val else "Decision Tree"
print("Best-performing model on validation data:", best_name)


RMSE Linear Regression (validation): 3.5923
RMSE Decision Tree (validation):     99.3594
Best-performing model on validation data: Linear Regression


In [17]:
# f) Retrain the best-performing model on training + validation data combined
best_model = LinearRegression() if best_name == "Linear Regression" else DecisionTreeRegressor(random_state=42)
best_model.fit(X_train_full, y_train_full)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](5,)","[77.3 ,87.95,71.89,28.57,31.61]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](5,)","['x1','x2','x3','x4','x5']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,0.1593
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,5


In [18]:
# g) Evaluate the model on the test data
rmse_test = root_mean_squared_error(y_test, best_model.predict(X_test))
print(f"RMSE {best_name} (test): {rmse_test:.4f}")

RMSE Linear Regression (test): 3.3715


In [19]:
# h) Retrain the model on the entire dataset (ready for production use)
best_model.fit(X, y)
best_model

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](5,)","[77.24,87.85,71.77,28.45,31.66]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Defined only when `X`has feature names that are all strings... versionadded:: 1.0","ndarray[object](5,)","['x1','x2','x3','x4','x5']"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,0.1269
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,5
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int,5


**Interpretation:** Linear Regression clearly outperforms the Decision Tree on this dataset (much lower validation RMSE), which makes sense given how `x1`–`x5` relate to `target` in this data — it behaves like a linear/near-linear relationship with added noise. This mirrors exactly the seven-step-checklist workflow from Question 1 (train → validate → pick winner → retrain on train+val → test → retrain on everything) and the "don't touch the test data until the very end" principle from Question 5.

## 10. The dataset `salary_dataset.csv` contains 29 observations of people. `YearsExperience` is how many years they've worked, and `Salary` is their salary.

a) Read the dataset `salary_dataset.csv` with pandas' `read_csv()` function and split it into X and y. Decide for yourself which variable should be the dependent variable y.
b) Split the dataset into a training and a test set. (So no validation set!)
c) Train two regression models with k-fold cross-validation using the `cross_validate()` function from scikit-learn. Use `neg_root_mean_squared_error` as scoring. Choose for yourself how many iterations it should do via the `cv` hyperparameter.
d) Evaluate the model that performs best on the test set.

Following Chapter 1's convention (Section 1.2, p. 17: "det är rimligt att tänka sig att åldern påverkar inkomst och inte tvärtom" — it's natural to think age affects income and not the other way around), it is `Salary` that depends on `YearsExperience`, not the reverse, so `y = Salary` and `X = YearsExperience`.